# April 24/04 - Python Functions

In [17]:
"""
Q1. Data Read Function (Extraction)
Write function to read data from transactions file. 
def read_data():
    return transactions
"""
import json

def read_data():
    with open("raw_transactions.json", "r") as file:
        data = json.load(file)
    return data

data = read_data()
print(data[:3])

[{'id': 1, 'customer': 'A', 'amount': 100, 'status': 'SUCCESS'}, {'id': 2, 'customer': 'B', 'amount': '200', 'status': 'FAILED'}, {'id': 3, 'customer': 'A', 'amount': None, 'status': 'SUCCESS'}]


In [18]:
"""
Q2. Validation Function: 
Write function to separate valid vs invalid records. 
Valid if:
	status == "SUCCESS"
	amount is not None / "" / "NULL"
	amount is numeric (or convertible)
Return: True / False
Note:
Expected Approach:
	Use conditions
	Use try-except (specific)
Don’t mix transformation here

"""

def is_valid(record):
    
    if record["status"] != "SUCCESS":
        return False
    
    amount = record["amount"]
    
    if amount is None or amount == "" or amount == "NULL":
        return False
    
    try:
        float(amount)
    except:
        return False
    
    return True


data = read_data()

valid_records = []
invalid_records = []

for r in data:
    if is_valid(r):
        valid_records.append(r)
    else:
        invalid_records.append(r)

print("Valid count:", len(valid_records))
print("Invalid count:", len(invalid_records))

Valid count: 19
Invalid count: 11


In [19]:
"""
Q3. Write transformation Function for: 
Convert amount to int/float
Keep only: {"customer": ..., "amount": ...} 
	(Schema Standardization)

Expected Approach
	Copy data (don’t mutate original)
	Convert safely
"""
def transform_data(records):
    result = []
    
    for r in records:
        new_record = {
            "customer": r["customer"],
            "amount": float(r["amount"])
        }
        result.append(new_record)
    
    return result


clean_data = transform_data(valid_records)
print(clean_data[:5])

[{'customer': 'A', 'amount': 100.0}, {'customer': 'C', 'amount': 300.0}, {'customer': 'A', 'amount': 100.0}, {'customer': 'D', 'amount': -50.0}, {'customer': 'E', 'amount': 500.5}]


# April 25/04: Python Functions: 

In [20]:
"""
Q4. Business Filter Function
Write function to filter 'amount > 0' : 
	def is_valid_business(record):
		...
Real-World Context: Remove irrelevant data (e.g., refunds, negatives)

"""
def is_valid_business(record):
    if record["amount"] > 0:
        return True
    else:
        return False


business_valid = []

for r in clean_data:
    if is_valid_business(r):
        business_valid.append(r)

print(business_valid[:5])

[{'customer': 'A', 'amount': 100.0}, {'customer': 'C', 'amount': 300.0}, {'customer': 'A', 'amount': 100.0}, {'customer': 'E', 'amount': 500.5}, {'customer': 'B', 'amount': 150.0}]


In [21]:
"""
Q5. Load Function (Aggregation)
Write function to load data. 
	def load_data(records):
		...

Output:
	{
		"A": total_amount,
		"B": total_amount,
		...
	}
Note: 
	Use dictionary
	Use .get()
"""

def load_data(records):
    result = {}
    
    for r in records:
        customer = r["customer"]
        amount = r["amount"]
        
        result[customer] = result.get(customer, 0) + amount
    
    return result


final_output = load_data(business_valid)
print(final_output)

{'A': 750.0, 'C': 1050.0, 'E': 1100.5, 'B': 350.0, 'D': 700.0, 'F': 750.0}


In [22]:
"""
Q6. Combine All (Mini ETL Pipeline)
Write function to combine/call all functions above: 
	def process_pipeline():
		...

Flow:
	Read data
	Validate
	Transform
	Apply business rule
	Aggregate
"""
def process_pipeline():

    data = read_data()
    
    valid = []
    for r in data:
        if is_valid(r):
            valid.append(r)
    
    transformed = transform_data(valid)
    
    business = []
    for r in transformed:
        if is_valid_business(r):
            business.append(r)
    
    result = load_data(business)
    
    return result


output = process_pipeline()
print(output)

{'A': 750.0, 'C': 1050.0, 'E': 1100.5, 'B': 350.0, 'D': 700.0, 'F': 750.0}
